In [1]:
# ============ Phase 1.1 — Load tokenizer and model ============
# What changes vs nothing (this is the first step):
#   We obtain two Python objects: `tokenizer` and `model`.
# New tensor introduced: none yet (no forward pass run).
# New static data introduced: all of the model's learned weights are now in RAM,
#   but we do not inspect them in this step.

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,   # decision #12: bf16 on CPU
    device_map="cpu",             # decision #3:  CPU-only
)
model.eval()                      # disable dropout; we are not training


# --- observation block ---
def human_bytes(n):
    for unit in ["B", "KB", "MB", "GB"]:
        if n < 1024:
            return f"{n:.2f} {unit}"
        n /= 1024
    return f"{n:.2f} TB"

n_params = sum(p.numel() for p in model.parameters())
weight_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
first_param = next(model.parameters())

print("=" * 50)
print(" TOKENIZER")
print("=" * 50)
print(f"  class       : {type(tokenizer).__name__}")
print(f"  vocab size  : {tokenizer.vocab_size:,}")
print(f"  model_max_len: {tokenizer.model_max_length:,}")

print()
print("=" * 50)
print(" MODEL")
print("=" * 50)
print(f"  class       : {type(model).__name__}")
print(f"  dtype       : {first_param.dtype}")
print(f"  device      : {first_param.device}")
print(f"  param count : {n_params:,}  (~{n_params/1e9:.2f} B)")
print(f"  weight RAM  : {human_bytes(weight_bytes)}")
print(f"  training?   : {model.training}")
print("=" * 50)


# --- assertion block ---
assert 1.0e9 < n_params < 1.5e9, f"unexpected param count: {n_params}"
assert first_param.dtype == torch.bfloat16, "weights not in bf16"
assert model.training is False, "model should be in eval() mode"

print("\n[OK] all assertions passed — Phase 1.1 complete")

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

 TOKENIZER
  class       : TokenizersBackend
  vocab size  : 128,000
  model_max_len: 131,072

 MODEL
  class       : LlamaForCausalLM
  dtype       : torch.bfloat16
  device      : cpu
  param count : 1,235,814,400  (~1.24 B)
  weight RAM  : 2.30 GB
  training?   : False

[OK] all assertions passed — Phase 1.1 complete


In [2]:
# ============ Phase 1.2 — Map document symbols to concrete numbers ============
# What changes vs 1.1: we read model.config and bind 8 size variables.
# New tensor introduced: none.
# New static data introduced: none — config is just integers.

cfg = model.config

# --- raw config dump (look at it once) ---
print("=" * 60)
print(" RAW CONFIG (relevant fields only)")
print("=" * 60)
print(f"  hidden_size              : {cfg.hidden_size}")
print(f"  num_hidden_layers        : {cfg.num_hidden_layers}")
print(f"  num_attention_heads      : {cfg.num_attention_heads}")
print(f"  num_key_value_heads      : {cfg.num_key_value_heads}")
print(f"  vocab_size               : {cfg.vocab_size:,}")
print(f"  intermediate_size        : {cfg.intermediate_size:,}")
print(f"  max_position_embeddings  : {cfg.max_position_embeddings:,}")

# --- bind to document symbols ---
d          = cfg.hidden_size
N          = cfg.num_hidden_layers
n_h        = cfg.num_attention_heads
n_kv       = cfg.num_key_value_heads
vocab_size = cfg.vocab_size
d_ff       = cfg.intermediate_size
max_pos    = cfg.max_position_embeddings

d_h        = d // n_h          # derived: dimension per head

print()
print("=" * 60)
print(" DOCUMENT SYMBOLS → CONCRETE NUMBERS")
print("=" * 60)
print(f"  d          (hidden size)           = {d}")
print(f"  N          (num blocks)            = {N}")
print(f"  n_h        (num query heads)       = {n_h}")
print(f"  n_kv       (num key/value heads)   = {n_kv}")
print(f"  d_h        (dim per head, d/n_h)   = {d_h}")
print(f"  d_ff       (FFN hidden dim)        = {d_ff:,}")
print(f"  vocab_size                         = {vocab_size:,}")
print(f"  max_pos    (max seq length)        = {max_pos:,}")
print("=" * 60)

# --- assertions: structural invariants every transformer must satisfy ---
assert d % n_h == 0,    f"d ({d}) not divisible by n_h ({n_h})"
assert n_h % n_kv == 0, f"n_h ({n_h}) not divisible by n_kv ({n_kv}) — GQA requires this"
assert d_ff > d,        f"d_ff ({d_ff}) should be larger than d ({d})"

print("\n[OK] all structural invariants hold — Phase 1.2 complete")

 RAW CONFIG (relevant fields only)
  hidden_size              : 2048
  num_hidden_layers        : 16
  num_attention_heads      : 32
  num_key_value_heads      : 8
  vocab_size               : 128,256
  intermediate_size        : 8,192
  max_position_embeddings  : 131,072

 DOCUMENT SYMBOLS → CONCRETE NUMBERS
  d          (hidden size)           = 2048
  N          (num blocks)            = 16
  n_h        (num query heads)       = 32
  n_kv       (num key/value heads)   = 8
  d_h        (dim per head, d/n_h)   = 64
  d_ff       (FFN hidden dim)        = 8,192
  vocab_size                         = 128,256
  max_pos    (max seq length)        = 131,072

[OK] all structural invariants hold — Phase 1.2 complete


In [3]:
# ============ Phase 1.3 — Micro-check: verify size relationships ============
# What changes vs 1.2: no new objects, just verifying predictions.

print("=" * 60)
print(" PREDICTION CHECKS")
print("=" * 60)

# Q1 — corrected: the meaningful relation is n_h * d_h == d
q_output_dim = n_h * d_h
print(f"Q1. n_h * d_h = {n_h} * {d_h} = {q_output_dim}")
print(f"    Should equal d = {d}     →  {'YES' if q_output_dim == d else 'NO'}")

# Q2 — total KV dimension per token (per block)
kv_output_dim = n_kv * d_h
print(f"\nQ2. n_kv * d_h = {n_kv} * {d_h} = {kv_output_dim}")
print(f"    This is the total K (or V) dim per token in one block.")
print(f"    Compare to Q dim = {q_output_dim}")
print(f"    Q is {q_output_dim // kv_output_dim}x larger than K/V  → GQA savings")

# Q3 — FFN expansion factor
expansion = d_ff / d
print(f"\nQ3. d_ff / d = {d_ff} / {d} = {expansion:.1f}")
print(f"    FFN expands width by {int(expansion)}x, then contracts back.")

# --- assertions ---
assert n_h * d_h == d,        f"n_h * d_h ({n_h*d_h}) != d ({d})"
assert n_kv * d_h < d,        f"GQA broken: n_kv * d_h should be < d"
assert d_ff == 4 * d,         f"d_ff ({d_ff}) is not 4x d ({d})"

print("\n" + "=" * 60)
print(" KEY RELATIONS TO REMEMBER")
print("=" * 60)
print(f"  n_h * d_h  = d              ({n_h} * {d_h} = {d})")
print(f"  n_kv * d_h = d / 4          ({n_kv} * {d_h} = {n_kv*d_h})")
print(f"  d_ff       = 4 * d          ({d_ff} = 4 * {d})")
print(f"  n_h / n_kv = 4              (each KV head shared by 4 Q heads)")
print("=" * 60)

print("\n[OK] Phase 1.3 complete")

 PREDICTION CHECKS
Q1. n_h * d_h = 32 * 64 = 2048
    Should equal d = 2048     →  YES

Q2. n_kv * d_h = 8 * 64 = 512
    This is the total K (or V) dim per token in one block.
    Compare to Q dim = 2048
    Q is 4x larger than K/V  → GQA savings

Q3. d_ff / d = 8192 / 2048 = 4.0
    FFN expands width by 4x, then contracts back.

 KEY RELATIONS TO REMEMBER
  n_h * d_h  = d              (32 * 64 = 2048)
  n_kv * d_h = d / 4          (8 * 64 = 512)
  d_ff       = 4 * d          (8192 = 4 * 2048)
  n_h / n_kv = 4              (each KV head shared by 4 Q heads)

[OK] Phase 1.3 complete
